**Objetivo del notebook:**

a. Integrar las bases de datos de casos no fatales y de muerte, en una sola base de datos. En la cual se identifica, cada uno de los casos que integran los 4 clusters en: peligro bajo (1), moderado (2), grave(0) y extremo (-1) de muerte por VIF o VP. Cada uno de estos grupos tiene un tamaño de 298774, 176830, 167687, 101 casos respectivamente.

b. Determinar experimentalmente el mejor equilibrio entre la detección de los casos de peligro extremo y el desempeño global del modelo, teniendo en cuenta que el tamaño del grupo peligro extremo, es mucho más pequeño que los otros grupos. El grupo mas grande que corresponde al peligro bajo es 2958 veces mas grande que el grupo peligro extremo. 

c. Dividir la base de datos en entrenamiento y prueba, construir los escenarios de balanceo y guardar los conjuntos de datos como tablas Delta.

-Entrada: 
            
            Bases: 
            
                    nofatales_clusterizados , origen: notebook 03_2

                    nofatales_muerte_var_significativas , origen: notebook 01_7

-Salida:
            Bases: 

                    dataset_modelado

                    train_original

                    test

                    train_balanceado_500

                    train_balanceado_1000

                    train_balanceado_2000

            Tabla:

                    escenarios_balanceo

In [0]:
%pip install imbalanced-learn

In [0]:
%restart_python

In [0]:

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import count, lit, col, when, regexp_replace, concat, round
from pyspark.sql.types import LongType

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt

# K-Modes
#from kmodes.kmodes import KModes

# Tiempo de ejecución
import time

# Ignorar advertencias
import warnings
warnings.filterwarnings("ignore")

#sklearn
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

In [0]:
#Cargar tabla de los casos no fatales agrupados en los 3 clusteres

df_nofatales = spark.table(
    "ml_proyecto_7405607705157039.default.nofatales_clusterizados"
)

In [0]:
#Cargar tabla de los casos de muerte

df_base = spark.table(
    "ml_proyecto_7405607705157039.default.nofatales_muerte_var_significativas"
)

df_muerte = df_base.filter(col("grupo")=="muerte")

In [0]:
#verificar el número de registros.

print("No fatales:", df_nofatales.count())
print("Muertes:", df_muerte.count())

In [0]:
#verificar columnas
df_nofatales.columns

In [0]:
#verificar columnas
df_muerte.columns

In [0]:
#esquema
df_nofatales.printSchema()

In [0]:
df_nofatales.display()

In [0]:
#esquema
df_muerte.printSchema()

In [0]:
#Eliminar la columna grupo del conjunto de no fatales

df_nofatales = df_nofatales.drop("grupo")

In [0]:
df_nofatales.display()

In [0]:
#Eliminar la columna grupo del conjunto de muertes

df_muerte = df_muerte.drop("grupo")

In [0]:
#Crear la columna cluster en el conjunto de muertes

df_muerte = df_muerte.withColumn(
    "cluster",
    F.lit(-1).cast(LongType())
)

In [0]:
df_muerte.display()

In [0]:
df_nofatales.printSchema()

In [0]:
df_muerte.printSchema()

**Construcción de la variable objetivo**

In [0]:
#Crear la variable nivel_peligro en df_nofatales

df_nofatales = (
    df_nofatales
    .withColumn(
        "nivel_peligro",
        when(F.col("cluster") == 0, "Peligro grave")
        .when(F.col("cluster") == 2, "Peligro moderado")
        .when(F.col("cluster") == 1, "Peligro bajo")
    )
)

In [0]:
df_nofatales.printSchema()

In [0]:
#Crear la variable nivel_peligro en df_muerte

df_muerte = (
    df_muerte
    .withColumn(
        "nivel_peligro",
        F.lit("Peligro extremo")
    )
)

In [0]:
#verificar asignación

df_nofatales.select( "cluster", "nivel_peligro").distinct().orderBy("cluster").show()

In [0]:
#verificar asignación

df_muerte.select("cluster","nivel_peligro").distinct().show()

In [0]:
#Integrar ambos conjuntos de datos

df_modelado = df_nofatales.unionByName(df_muerte)

In [0]:
#Verificar la integración
print("Total registros:", df_modelado.count())

In [0]:
#Revisar distribución de los niveles de peligro

display(
    df_modelado
    .groupBy("nivel_peligro")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
#verificar que los clusters se conservaron correctamente

display(
    df_modelado
    .groupBy("cluster", "nivel_peligro")
    .count()
    .orderBy("cluster")
)

In [0]:
#Guardar datos en tabla Delta 

(
    df_modelado
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.dataset_modelado"
    )
)

**Análisis de la distribución de clases**

In [0]:
#Distribución absoluta

display(
    df_modelado
    .groupBy("nivel_peligro")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
#Distribución porcentual

total = df_modelado.count()

df_distribucion = (
    df_modelado
    .groupBy("nivel_peligro")
    .count()
    .withColumn(
        "porcentaje",
        round(col("count") * 100 / total, 4)
    )
    .orderBy(F.desc("count"))
)

display(df_distribucion)

Databricks visualization. Run in Databricks to view.

**División estratificada en entrenamiento y prueba**

Dividir el conjunto de datos en un conjunto de entrenamiento (80 %) y un conjunto de prueba (20 %), preservando la proporción de cada nivel de peligro. Esta partición se realiza antes de aplicar cualquier técnica de balanceo para garantizar una evaluación objetiva del desempeño de los modelos supervisados.

In [0]:
#Verificar la distribución inicial

#Antes de dividir los datos, verificamos nuevamente la distribución de la variable objetivo

display(
    df_modelado
    .groupBy("nivel_peligro")
    .count()
    .orderBy("nivel_peligro")
)


**Dividir estratificadamente**

La partición del conjunto de datos se realiza mediante un muestreo estratificado con una proporción de 80 % para entrenamiento y 20 % para prueba, preservando la distribución original de los niveles de peligro. De esta forma, el conjunto de prueba mantiene las características reales del problema y permite evaluar el desempeño de los modelos en condiciones representativas. A partir de este punto, cualquier técnica de balanceo se aplicará exclusivamente sobre el conjunto de entrenamiento, evitando fuga de información hacia el conjunto de prueba.

In [0]:
#Dividir estratificadamente los datos de entrenamiento y de prueba con scikit-learn

#convertir la base de datos de spark a pandas
df_modelado_pd = df_modelado.toPandas() 


In [0]:
train_pd, test_pd = train_test_split(
    df_modelado_pd,
    test_size=0.20,
    stratify=df_modelado_pd["nivel_peligro"],
    random_state=42
)

In [0]:
#Regresar a Spark

train = spark.createDataFrame(train_pd)

test = spark.createDataFrame(test_pd)

In [0]:
#verificación distribución de casos por nivel_de_peligro en data de entrenamiento
display(
    train
    .groupBy("nivel_peligro")
    .count()
    .orderBy("nivel_peligro")
)

In [0]:
#verificación distribución de casos por nivel_de_peligro en data de prueba
display(
    test
    .groupBy("nivel_peligro")
    .count()
    .orderBy("nivel_peligro")
)

In [0]:
#Guardar como Delta

(
    train.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.train_original"
    )
)

In [0]:
(
    test.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.test"
    )
)

In [0]:
#Validación de la estratificación

# Distribución porcentual en entrenamiento
total_train = train.count()

display(
    train.groupBy("nivel_peligro")
         .count()
         .withColumn(
             "porcentaje",
             F.round(F.col("count") * 100 / total_train, 4)
         )
         .orderBy("nivel_peligro")
)

In [0]:
# Distribución porcentual en prueba
total_test = test.count()

display(
    test.groupBy("nivel_peligro")
        .count()
        .withColumn(
            "porcentaje",
            F.round(F.col("count") * 100 / total_test, 4)
        )
        .orderBy("nivel_peligro")
)

**Construcción de los escenarios de balanceo**

Debido al marcado desbalance de la variable objetivo, se construyeron diferentes escenarios experimentales para evaluar el efecto del remuestreo sobre el desempeño de los modelos de clasificación.

El balanceo se realizará únicamente sobre el conjunto de entrenamiento, mientras que el conjunto de prueba conservará la distribución original de las clases.

Inicialmente se utilizará una estrategia de sobremuestreo aleatorio (RandomOverSampler), la cual incrementa la representación de la clase minoritaria sin generar observaciones sintéticas. Posteriormente, durante el entrenamiento de los modelos supervisados, se incorporarán pesos de clase para penalizar en mayor medida los errores sobre la clase Peligro extremo.

In [0]:
#Leer el conjunto de entrenamiento

train = spark.table(
    "ml_proyecto_7405607705157039.default.train_original"
)

In [0]:
#Convertir a Pandas

train_pd = train.toPandas()

In [0]:
#Verificar distribución inicial

train_pd["nivel_peligro"].value_counts()

In [0]:
#Separar variables predictoras y variable objetivo

X_train = train_pd.drop(columns=["nivel_peligro"])

y_train = train_pd["nivel_peligro"]

In [0]:
#Escenario 1 (500 casos)

ros_500 = RandomOverSampler(
    sampling_strategy={
        "Peligro extremo": 500
    },
    random_state=42
)

X_500, y_500 = ros_500.fit_resample(
    X_train,
    y_train
)

In [0]:
pd.Series(y_500).value_counts()

In [0]:
#Escenario 2 (1000 casos)

ros_1000 = RandomOverSampler(
    sampling_strategy={
        "Peligro extremo": 1000
    },
    random_state=42
)

X_1000, y_1000 = ros_1000.fit_resample(
    X_train,
    y_train
)

In [0]:
pd.Series(y_1000).value_counts()

In [0]:
ros_2000 = RandomOverSampler(
    sampling_strategy={
        "Peligro extremo": 2000
    },
    random_state=42
)

X_2000, y_2000 = ros_2000.fit_resample(
    X_train,
    y_train
)

In [0]:
pd.Series(y_2000).value_counts()

In [0]:
#Reconstruir los DataFrames

#Escenario 500

train_500 = X_500.copy()

train_500["nivel_peligro"] = y_500

In [0]:
#Escenario 1000

train_1000 = X_1000.copy()

train_1000["nivel_peligro"] = y_1000

In [0]:
# Escenario 2000

train_2000 = X_2000.copy()

train_2000["nivel_peligro"] = y_2000

In [0]:
#Convertir nuevamente a Spark

train_500_spark = spark.createDataFrame(train_500)

train_1000_spark = spark.createDataFrame(train_1000)

train_2000_spark = spark.createDataFrame(train_2000)

In [0]:
#Guardar como Delta

(
    train_500_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.train_balanceado_500"
    )
)

In [0]:
(
    train_1000_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.train_balanceado_1000"
    )
)

In [0]:
(
    train_2000_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.train_balanceado_2000"
    )
)

**Construcción de la tabla resumen de escenarios**

In [0]:
#Función para resumir un escenario

def resumen_escenario(df, nombre_escenario):
    """
    Genera una fila resumen con la distribución de clases
    para un escenario de entrenamiento.
    """

    resumen = (
        df.groupBy("nivel_peligro")
          .count()
          .groupBy()
          .pivot("nivel_peligro")
          .agg(F.first("count"))
          .fillna(0)
    )

    # Garantizar que existan todas las columnas
    clases = [
        "Peligro extremo",
        "Peligro grave",
        "Peligro moderado",
        "Peligro bajo"
    ]

    for clase in clases:
        if clase not in resumen.columns:
            resumen = resumen.withColumn(clase, F.lit(0))

    resumen = resumen.select(
        *clases
    )

    resumen = (
        resumen
        .withColumn("escenario", F.lit(nombre_escenario))
        .withColumn(
            "total_registros",
            F.col("Peligro extremo") +
            F.col("Peligro grave") +
            F.col("Peligro moderado") +
            F.col("Peligro bajo")
        )
    )

    return resumen

In [0]:
#Crear el resumen de cada escenario

esc_original = resumen_escenario(train, "Original")

esc_500 = resumen_escenario(
    train_500_spark,
    "Balanceado_500"
)

esc_1000 = resumen_escenario(
    train_1000_spark,
    "Balanceado_1000"
)

esc_2000 = resumen_escenario(
    train_2000_spark,
    "Balanceado_2000"
)

In [0]:
#Unir todos los escenarios

df_escenarios = (
    esc_original
    .unionByName(esc_500)
    .unionByName(esc_1000)
    .unionByName(esc_2000)
)

In [0]:
#Calcular el incremento de la clase minoritaria

casos_original = (
    df_escenarios
    .filter(F.col("escenario") == "Original")
    .select("Peligro extremo")
    .first()[0]
)

In [0]:
#calcular el incremento.

df_escenarios = (
    df_escenarios
    .withColumn(
        "incremento_peligro_extremo",
        F.col("Peligro extremo") - F.lit(casos_original)
    )
)

In [0]:
#Ordenar las columnas

df_escenarios = df_escenarios.select(
    "escenario",
    "Peligro extremo",
    "Peligro grave",
    "Peligro moderado",
    "Peligro bajo",
    "total_registros",
    "incremento_peligro_extremo"
)

In [0]:
#Visualizar

display(df_escenarios)

In [0]:
df_escenarios_limpio = (
    df_escenarios
    .withColumnRenamed("Peligro extremo", "peligro_extremo")
    .withColumnRenamed("Peligro grave", "peligro_grave")
    .withColumnRenamed("Peligro moderado", "peligro_moderado")
    .withColumnRenamed("Peligro bajo", "peligro_bajo")
)

In [0]:
df_escenarios_limpio.printSchema()

In [0]:
#Guardar como tabla Delta

(
    df_escenarios_limpio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.escenarios_balanceo"
    )
)